In [7]:
import pandas as pd
import numpy as np
import os

from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    accuracy_score
)

import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

In [9]:
try:
    import shap
    print("[INFO] SHAP library found")
except ImportError:
    shap = None
    print("[INFO] SHAP library not found, skipping SHAP-related features")

print("All libraries imported successfully")

[INFO] SHAP library found
All libraries imported successfully


In [10]:
csv_file = "factory_sensor_simulator_2040.csv" 
print(f"[INFO] Loading: {csv_file}")

try:
    df = pd.read_csv(csv_file)
    print(f"[INFO] Loaded shape: {df.shape}")
    target_column = 'Failure_Within_7_Days'
    drop_cols = ['Machine_ID', 'Machine_Type', 'Remaining_Useful_Life_days', target_column]
    X = df.drop(drop_cols, axis=1, errors='ignore')
    if 'AI_Supervision' in X.columns:
        X['AI_Supervision'] = X['AI_Supervision'].astype(int)
    y = df[target_column].astype(int)
    FEATURES_KEY = X.columns.tolist()
    print(f"[INFO] Features used: {FEATURES_KEY}")
except FileNotFoundError:
    print(f"[ERROR] File not found: {csv_file}")
except Exception as e:
    print(f"[ERROR] {e}")

[INFO] Loading: factory_sensor_simulator_2040.csv
[INFO] Loaded shape: (500000, 22)
[INFO] Features used: ['Installation_Year', 'Operational_Hours', 'Temperature_C', 'Vibration_mms', 'Sound_dB', 'Oil_Level_pct', 'Coolant_Level_pct', 'Power_Consumption_kW', 'Last_Maintenance_Days_Ago', 'Maintenance_History_Count', 'Failure_History_Count', 'AI_Supervision', 'Error_Codes_Last_30_Days', 'Laser_Intensity', 'Hydraulic_Pressure_bar', 'Coolant_Flow_L_min', 'Heat_Index', 'AI_Override_Events']


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("[INFO] data split complete.")
print(f"Total Samples: {len(y)}")
print(f"Training Samples: {len(y_train)}")
print(f"Testing Samples: {len(y_test)}")

[INFO] data split complete.
Total Samples: 500000
Training Samples: 400000
Testing Samples: 100000


In [12]:
print("[INFO] Imputing missing values...")
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

[INFO] Imputing missing values...


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)
print("[INFO] Data scaling complete")

[INFO] Data scaling complete


In [19]:
print("[INFO] Training RandomForest...")
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Helps the model focus more on catching the rare failure events
)
rf.fit(X_train_scaled, y_train)
print("[INFO] RandomForest training complete")

[INFO] Training RandomForest...
[INFO] RandomForest training complete


In [20]:
print("[INFO] Evaluating model on test data")
y_pred = rf.predict(X_test_scaled)

try:
    y_score = rf.predict_proba(X_test_scaled)[:, 1]
except Exception:
    y_score = None
    print("[WARN] could not get predict_proba scores")

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='binary')
rec = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')
cm = confusion_matrix(y_test, y_pred)

# Print results
print(f"[INFO] Accuracy  : {acc:.4f}")
print(f"[INFO] Precision : {prec:.4f}")
print(f"[INFO] Recall    : {rec:.4f}")
print(f"[INFO] F1 Score  : {f1:.4f}")

print("\n[RESULT] Confusion Matrix:")
# Changed index and column names to represent operational statuses
print(pd.DataFrame(cm, index=["Actual Normal", "Actual Failure"], columns=["Pred. Normal", "Pred. Failure"]))

print("\n[RESULT] Classification Report:")
print(classification_report(y_test, y_pred))

[INFO] Evaluating model on test data
[INFO] Accuracy  : 0.9615
[INFO] Precision : 0.7176
[INFO] Recall    : 0.5924
[INFO] F1 Score  : 0.6490

[RESULT] Confusion Matrix:
                Pred. Normal  Pred. Failure
Actual Normal          92594           1400
Actual Failure          2448           3558

[RESULT] Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     93994
           1       0.72      0.59      0.65      6006

    accuracy                           0.96    100000
   macro avg       0.85      0.79      0.81    100000
weighted avg       0.96      0.96      0.96    100000



In [21]:
roc_data, pr_data = {}, {}
if y_score is not None:
    try:
        fpr, tpr, _ = roc_curve(y_test, y_score)
        precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_score)
        roc_data = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}
        pr_data = {"precision": precision_curve.tolist(), "recall": recall_curve.tolist()}
        print("[INFO] ROC and PR curve data calculated")
    except Exception as e:
        print(f"Could not compute ROC/PR data: {e}")
else:
    print("[INFO] Skipping ROC/PR calculation (no probability scores)")

[INFO] ROC and PR curve data calculated


In [22]:
top_features = []
try:
    importances = rf.feature_importances_

    # Using the FEATURES_KEY defined during data loading
    feat_imp = sorted(
        zip(FEATURES_KEY, importances),
        key=lambda x: x[1],
        reverse=True
    )[:10]

    top_features = [
        {"feature": f, "importance": float(imp)}
        for f, imp in feat_imp
    ]

    print("\n[INFO] Top important features")
    print("." * 40)
    for f, imp in feat_imp:
        print(f"{f:<30s} : {imp:.6f}")
    print("." * 40)

except Exception as e:
    print(f"[WARN] Could not extract important features: {e}")


[INFO] Top important features
........................................
Operational_Hours              : 0.843067
Vibration_mms                  : 0.018561
Temperature_C                  : 0.017566
Power_Consumption_kW           : 0.015173
Sound_dB                       : 0.014874
Last_Maintenance_Days_Ago      : 0.014250
Oil_Level_pct                  : 0.013992
Coolant_Level_pct              : 0.013461
Installation_Year              : 0.011157
Maintenance_History_Count      : 0.007934
........................................


In [23]:
output_dir = "models"
os.makedirs(output_dir, exist_ok=True)

try:
    # joblib.dump(imputer, os.path.join(output_dir, "factory_sensor-imputer.joblib"))
    joblib.dump(scaler, os.path.join(output_dir, "factory_sensor_simulator_scaler.joblib"))
    joblib.dump(rf, os.path.join(output_dir, "factory_sensor_simulator_rf.joblib"))
    
    if 'explainer' in locals():
        joblib.dump(explainer, os.path.join(output_dir, "shap_explainer.joblib"))

    print(f"[INFO] All models, imputer, and scaler saved to '{output_dir}/' directory.")
except Exception as e:
    print(f"[ERROR] Could not save the models: {e}")

[INFO] All models, imputer, and scaler saved to 'models/' directory.
